# 🔬 AI vs Real Image Detection - Pengujian Model Berskala Besar pada Dataset Thunderbeast

Notebook ini dirancang untuk melakukan **pengujian (testing) dan evaluasi performa model secara mendalam dengan skala data besar** menggunakan dataset resmi: **`Thunderbeast/ai-generated-vs-real-images-datasets`** yang tersimpan di direktori cache lokal `./data`.

---

### 📌 Karakteristik Pengujian:
- **Dataset Target**: `Thunderbeast/ai-generated-vs-real-images-datasets` (Tersimpan di `./data`).
- **Data Uji (Test Split)**: Menggunakan split validasi/testing 10% (~15.271 gambar uji) yang dipisahkan secara seimbang (*stratified*, `seed=42`).
- **Skala Pengujian Cepat**: Didukung PyTorch **`DataLoader` (Batch Size: 64)** dengan akselerasi GPU CUDA, mampu menguji **1.000+ gambar hanya dalam ~15-20 detik**!
- **Label Standar Proyek**:
  - `0: AiArtData` $ightarrow$ **AI-Generated (Kelas 0)**
  - `1: RealArt`   $ightarrow$ **Real Image (Kelas 1)**

---

### 🎯 Informasi yang Dihasilkan pada Pengujian:
1. **Hasil Prediksi**: `AI-Generated` atau `Real Image`.
2. **Tingkat Keyakinan (Confidence Score %)**: Nilai kepastian model terhadap keputusannya.
3. **Probabilitas Detail**: Persentase `P(AI) (%)` vs `P(Real) (%)`.
4. **Verifikasi Ground Truth**: Komparasi dengan label asli (Status: **`BENAR`** atau **`SALAH`**).
5. **Visualisasi Galeri Skala Besar**:
   - Galeri grid 24 sampel acak dengan border warna status (Hijau = Benar, Merah = Salah).
   - Galeri analisis kasus salah prediksi (*Misclassified Samples / Hard Cases*).
   - Galeri deteksi dengan keyakinan tertinggi (*High-Confidence Samples*).
   - Evaluasi statistik komprehensif: Confusion Matrix, Classification Report, dan Histogram Sebaran Confidence pada 1.000 citra uji.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*Palette images with Transparency.*')
warnings.filterwarnings('ignore', message='.*xFormers is not available.*')
warnings.filterwarnings('ignore', category=UserWarning)

import os
import time
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm

# Konfigurasi Hardware Acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f"✅ Akselerasi GPU Aktif: {device} ({torch.cuda.get_device_name(0)})")
    print("🚀 cuDNN Benchmark diaktifkan untuk throughput inferensi batch maksimal.")
else:
    print(f"ℹ️ Menggunakan CPU: {device}")

In [ ]:
print("=" * 75)
print("📦 MEMUAT DATASET THUNDERBEAST DARI CACHE LOKAL (./data)")
print("=" * 75)

# Memuat dataset Thunderbeast dari cache lokal tanpa unduh ulang
dataset = load_dataset(
    "Thunderbeast/AI-Generated-vs-Real-Images-Datasets",
    split="train",
    cache_dir="./data",
    verification_mode="no_checks"
)

# Pembagian data seimbang (90% Train, 10% Test/Validation) dengan seed=42
splits = dataset.train_test_split(test_size=0.10, stratify_by_column="label", seed=42)
test_dataset = splits["test"]

print(f"Total Seluruh Dataset    : {len(dataset):,} gambar")
print(f"Total Data Uji (Testing) : {len(test_dataset):,} gambar")
print(f"Daftar Kelas Asli        : {dataset.features['label'].names}")
print(f"  ├─ Kelas 0             : {dataset.features['label'].names[0]} (AI-Generated Image)")
print(f"  └─ Kelas 1             : {dataset.features['label'].names[1]} (Real Image)")
print("=" * 75)

In [ ]:
from model_custom import AIForensicNet

# Pipeline Transformasi Standar Evaluasi Model (224x224 px)
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Daftar model checkpoint yang didukung
MODELS_CATALOG = {
    "ConvNeXt-Tiny": {
        "paths": ["best_convnext_tiny_ai_vs_real.pth", "backup/best_convnext_tiny_ai_vs_real.pth"],
        "arch": "convnext_tiny",
        "name": "ConvNeXt-Tiny (Meta AI)"
    },
    "DINOv2-Small": {
        "paths": ["best_dinov2_vits14_ai_vs_real.pth", "backup/best_dinov2_vits14_ai_vs_real.pth"],
        "arch": "dinov2_vits14",
        "name": "DINOv2-Small (Meta AI)"
    },
    "EfficientNetV2-S": {
        "paths": ["best_efficientnet_v2_s_ai_vs_real.pth", "backup/best_efficientnet_v2_s_ai_vs_real.pth"],
        "arch": "efficientnet_v2_s",
        "name": "EfficientNetV2-S (Google)"
    },
    "AIForensicNet": {
        "paths": ["best_aiforensicnet_ai_vs_real.pth", "backup/best_aiforensicnet_ai_vs_real.pth"],
        "arch": "aiforensicnet",
        "name": "AIForensicNet (Custom Net)"
    },
    "EfficientNet-B0": {
        "paths": ["backup/best_efficientnet_b0_ai_vs_real.pth", "best_efficientnet_b0_ai_vs_real.pth"],
        "arch": "efficientnet_b0",
        "name": "EfficientNet-B0 (Baseline)"
    }
}

class DINOv2AppClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            hub_dir = os.path.expanduser("~/.cache/torch/hub/facebookresearch_dinov2_main")
            if os.path.exists(hub_dir):
                self.backbone = torch.hub.load(hub_dir, "dinov2_vits14", source="local")
            else:
                self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(self.backbone.embed_dim, 2)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


def build_model_architecture(arch: str) -> nn.Module:
    """Membangun arsitektur model sesuai tipe backbone."""
    if arch == "dinov2_vits14":
        return DINOv2AppClassifier()
    elif arch == "convnext_tiny":
        m = models.convnext_tiny(weights=None)
        in_f = m.classifier[2].in_features
        m.classifier[2] = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_f, 2)
        )
        return m
    elif arch == "efficientnet_v2_s":
        m = models.efficientnet_v2_s(weights=None)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(in_f, 2)
        )
        return m
    elif arch == "aiforensicnet":
        return AIForensicNet(num_classes=2)
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(in_f, 2)
        )
        return m
    else:
        raise ValueError(f"Arsitektur '{arch}' tidak didukung.")


def load_model_checkpoint(model_key="ConvNeXt-Tiny"):
    """Memuat model dan bobot checkpoint terbaik yang tersimpan."""
    if model_key not in MODELS_CATALOG:
        available = list(MODELS_CATALOG.keys())
        raise ValueError(f"Model '{model_key}' tidak ditemukan. Pilihan: {available}")
    
    cfg = MODELS_CATALOG[model_key]
    checkpoint_path = None
    for p in cfg["paths"]:
        if os.path.exists(p):
            checkpoint_path = p
            break
            
    if checkpoint_path is None:
        raise FileNotFoundError(f"File checkpoint '{model_key}' tidak ditemukan di {cfg['paths']}")
        
    print(f"📦 Memuat model: {cfg['name']}")
    print(f"📂 Checkpoint  : {checkpoint_path}")
    
    model = build_model_architecture(cfg["arch"])
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    print(f"✅ Model '{cfg['name']}' berhasil dimuat dan siap untuk pengujian!
")
    return model, cfg["name"]

# Inisialisasi Model Aktif (Default: ConvNeXt-Tiny)
model, active_model_name = load_model_checkpoint("ConvNeXt-Tiny")

In [ ]:
class ThunderbeastTestDataset(Dataset):
    """
    PyTorch Dataset wrapper berperforma tinggi untuk dataset Thunderbeast.
    Memungkinkan inferensi cepat dalam batch besar menggunakan PyTorch DataLoader.
    """
    def __init__(self, hf_dataset, max_samples=None, transform=eval_transforms, shuffle_data=True, seed=None):
        if shuffle_data:
            # Acak urutan dataset secara dinamis dari seluruh 15.271 gambar uji
            hf_dataset = hf_dataset.shuffle(seed=seed)
        if max_samples is not None and max_samples < len(hf_dataset):
            self.dataset = hf_dataset.select(range(max_samples))
        else:
            self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        label = item["label"]  # 0: AiArtData, 1: RealArt

        if image.mode == 'P' and 'transparency' in image.info:
            image = image.convert('RGBA')
        if image.mode != "RGB":
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label, idx

print("✅ PyTorch Dataset Wrapper dengan fitur Randomize siap digunakan.")

---
## 🚀 Bagian 1: Pengujian Skala Besar (1.000 Gambar Thunderbeast Teracak)

Di bawah ini kita melakukan inferensi pada **1.000 gambar uji Thunderbeast yang diacak (*randomized*)** dari total 15.271 data uji menggunakan `DataLoader` dengan `batch_size=64`.
Proses ini menghitung:
- **Hasil Prediksi**: AI-Generated (0) atau Real Image (1)
- **Tingkat Keyakinan (Confidence Score %)**
- **Distribusi Probabilitas**: P(AI) vs P(Real) (%)
- **Akurasi Keseluruhan & Evaluasi Metrik**

In [ ]:
# Konfigurasi Pengujian & Pengacakan Data
NUM_TEST_SAMPLES = 1000  # Jumlah sampel acak yang diuji (misal 500, 1000, 2000)
BATCH_SIZE = 64
RANDOMIZE_DATA = True    # Set True untuk mengacak data setiap kali dijalankan
RANDOM_SEED = None       # Ganti dengan integer (misal 42) jika ingin hasil acak yang konsisten/reproducible

print(f"🎲 Menyiapkan DataLoader untuk {NUM_TEST_SAMPLES:,} citra uji Thunderbeast (Randomized: {RANDOMIZE_DATA})...")
test_pytorch_ds = ThunderbeastTestDataset(
    test_dataset,
    max_samples=NUM_TEST_SAMPLES,
    shuffle_data=RANDOMIZE_DATA,
    seed=RANDOM_SEED
)
test_loader = DataLoader(test_pytorch_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

all_preds = []
all_labels = []
all_confs = []
all_probs_ai = []
all_probs_real = []
all_indices = []

start_time = time.time()

model.eval()
with torch.no_grad():
    for images, labels, indices in tqdm(test_loader, desc=f"Menguji {NUM_TEST_SAMPLES} Gambar"):
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        confs, preds = torch.max(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_confs.extend((confs.cpu().numpy() * 100.0))
        all_probs_ai.extend((probs[:, 0].cpu().numpy() * 100.0))
        all_probs_real.extend((probs[:, 1].cpu().numpy() * 100.0))
        all_indices.extend(indices.numpy())

elapsed_time = time.time() - start_time
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_confs = np.array(all_confs)
all_probs_ai = np.array(all_probs_ai)
all_probs_real = np.array(all_probs_real)
all_indices = np.array(all_indices)

# Hitung Akurasi & Rata-rata Confidence
accuracy = (all_preds == all_labels).mean() * 100.0
avg_conf = all_confs.mean()
correct_mask = (all_preds == all_labels)
avg_conf_correct = all_confs[correct_mask].mean()
avg_conf_wrong = all_confs[~correct_mask].mean() if (~correct_mask).sum() > 0 else 0.0

print("\n" + "=" * 70)
print(f"📊 HASIL EVALUASI BERSKALA BESAR PADA DATASET THUNDERBEAST")
print(f"Model: {active_model_name}")
print("=" * 70)
print(f"⏱️ Waktu Pengujian         : {elapsed_time:.2f} detik ({len(all_preds) / elapsed_time:.1f} gambar/detik)")
print(f"🖼️ Total Gambar Diuji      : {len(all_preds):,} gambar")
print(f"🎯 Akurasi Total            : {accuracy:.2f}% ({correct_mask.sum()}/{len(all_preds)} benar)")
print(f"📈 Rata-rata Confidence     : {avg_conf:.2f}%")
print(f"   ├─ Rata-rata saat Benar : {avg_conf_correct:.2f}%")
print(f"   └─ Rata-rata saat Salah : {avg_conf_wrong:.2f}%")
print("=" * 70)

# Classification Report
target_names = ["AI-Generated (0)", "Real Image (1)"]
print("\n📋 CLASSIFICATION REPORT:")
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

# Confusion Matrix & Histogram Sebaran Confidence
fig, (ax_cm, ax_hist) = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues', ax=ax_cm, values_format='d')
ax_cm.set_title(f'Confusion Matrix ({active_model_name})
{NUM_TEST_SAMPLES} Sampel Thunderbeast', fontsize=12, fontweight='bold')
ax_cm.grid(False)

# Histogram Sebaran Confidence
ax_hist.hist(all_confs[correct_mask], bins=15, color='#10B981', alpha=0.7, label=f'Prediksi Benar (Mean: {avg_conf_correct:.1f}%)')
if (~correct_mask).sum() > 0:
    ax_hist.hist(all_confs[~correct_mask], bins=15, color='#EF4444', alpha=0.7, label=f'Prediksi Salah (Mean: {avg_conf_wrong:.1f}%)')
ax_hist.axvline(avg_conf, color='#1E293B', linestyle='--', linewidth=2, label=f'Rata-rata: {avg_conf:.1f}%')
ax_hist.set_title(f'Distribusi Confidence Score ({NUM_TEST_SAMPLES} Gambar)', fontsize=12, fontweight='bold')
ax_hist.set_xlabel('Confidence (%)', fontsize=10, fontweight='bold')
ax_hist.set_ylabel('Jumlah Gambar', fontsize=10, fontweight='bold')
ax_hist.legend()
ax_hist.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

---
## 🖼️ Bagian 2: Galeri Visual Grid 24 Gambar Uji Thunderbeast

Berikut adalah visualisasi galeri dari **24 gambar uji pertama** dari dataset Thunderbeast.
Setiap gambar dilengkapi dengan:
- Border warna: **Hijau (Prediksi Benar)** atau **Merah (Prediksi Salah)**
- Judul berisi: **Ground Truth $\rightarrow$ Prediksi Model**, **Tingkat Keyakinan (Confidence %)**, dan **Status**.

In [ ]:
N_SHOW = 24
ncols = 6
nrows = 4

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 12.5))
axes = axes.flatten()

for i in range(N_SHOW):
    ax = axes[i]
    idx = all_indices[i]
    item = test_dataset[int(idx)]
    img = item["image"]

    is_correct = bool(correct_mask[i])
    border_col = '#10B981' if is_correct else '#EF4444'
    title_col = '#059669' if is_correct else '#DC2626'

    ax.imshow(img)
    ax.axis('off')

    # Border status
    rect = patches.Rectangle(
        (0, 0), img.width - 1, img.height - 1,
        linewidth=4, edgecolor=border_col, facecolor='none'
    )
    ax.add_patch(rect)

    true_s = "AI" if all_labels[i] == 0 else "Real"
    pred_s = "AI" if all_preds[i] == 0 else "Real"
    status_s = "BENAR" if is_correct else "SALAH"

    ax.set_title(
        f"#{idx} | {true_s} -> {pred_s} [{status_s}]
Conf: {all_confs[i]:.1f}%
AI: {all_probs_ai[i]:.1f}% | R: {all_probs_real[i]:.1f}%",
        color=title_col,
        fontsize=9,
        fontweight='bold',
        pad=5
    )

plt.suptitle(f"Galeri 24 Sampel Citra Thunderbeast ({active_model_name})", fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

---
## 🔍 Bagian 3: Analisis Kasus Menarik (Hard Samples vs High Confidence)

Sebagai evaluasi mendalam machine learning, kita menganalisis:
1. **Kasus Salah Prediksi (Misclassified Samples)**: Gambar seni yang gagal dikenali model atau membuat model tertipu, beserta persentase confidence-nya.
2. **Kasus Keyakinan Tertinggi (High-Confidence Predictions)**: Gambar yang dikenali dengan kepastian hampir 100%.

In [ ]:
# 1. Menemukan Sampel Salah Prediksi (Misclassified / Hard Cases)
wrong_indices = np.where(~correct_mask)[0]

print("=" * 75)
print(f"⚠️ TOTAL SAMPEL SALAH PREDIKSI: {len(wrong_indices)} dari {len(all_preds)} gambar ({len(wrong_indices)/len(all_preds)*100:.2f}%)")
print("=" * 75)

if len(wrong_indices) > 0:
    n_wrong_show = min(12, len(wrong_indices))
    ncols_w = 6
    nrows_w = int(np.ceil(n_wrong_show / ncols_w))
    fig, axes = plt.subplots(nrows_w, ncols_w, figsize=(18, 3.5 * nrows_w))
    axes = np.array(axes).reshape(-1)

    for i in range(n_wrong_show):
        w_pos = wrong_indices[i]
        idx = all_indices[w_pos]
        item = test_dataset[int(idx)]
        img = item["image"]

        ax = axes[i]
        ax.imshow(img)
        ax.axis('off')

        rect = patches.Rectangle(
            (0, 0), img.width - 1, img.height - 1,
            linewidth=4, edgecolor='#EF4444', facecolor='none'
        )
        ax.add_patch(rect)

        true_s = "AI-Generated" if all_labels[w_pos] == 0 else "Real Image"
        pred_s = "AI-Generated" if all_preds[w_pos] == 0 else "Real Image"

        ax.set_title(
            f"Index #{idx}
True: {true_s}
Pred: {pred_s}
Conf: {all_confs[w_pos]:.1f}%
(AI:{all_probs_ai[w_pos]:.1f}% | R:{all_probs_real[w_pos]:.1f}%)",
            color='#DC2626',
            fontsize=8.5,
            fontweight='bold',
            pad=5
        )

    for j in range(n_wrong_show, len(axes)):
        axes[j].axis('off')

    plt.suptitle("Analisis Gambar yang Salah Diprediksi (Hard / Challenging Samples)", fontsize=14, fontweight='bold', color='#DC2626')
    plt.tight_layout()
    plt.show()

# 2. Tabel Rangkuman 20 Sampel Acak
sample_table_indices = random.sample(range(len(all_preds)), 20)
table_data = []

for pos in sample_table_indices:
    idx = all_indices[pos]
    true_s = "AI-Generated" if all_labels[pos] == 0 else "Real Image"
    pred_s = "AI-Generated" if all_preds[pos] == 0 else "Real Image"
    is_ok = bool(correct_mask[pos])

    table_data.append({
        "Index": f"#{idx}",
        "Ground Truth": true_s,
        "Hasil Prediksi": pred_s,
        "Confidence (%)": f"{all_confs[pos]:.2f}%",
        "P(AI) (%)": f"{all_probs_ai[pos]:.2f}%",
        "P(Real) (%)": f"{all_probs_real[pos]:.2f}%",
        "Status": "✅ BENAR" if is_ok else "❌ SALAH"
    })

df_summary = pd.DataFrame(table_data)
print("\n" + "=" * 85)
print("TABEL SAMPEL EVALUASI DATASET THUNDERBEAST (20 Sampel Acak):")
print("=" * 85)
print(df_summary.to_string(index=False))
print("=" * 85)

---
## 🎲 Bagian 4: Pengujian Interaktif pada Sampel Acak Thunderbeast

Gunakan fungsi di bawah ini untuk menguji sejumlah gambar acak dari dataset Thunderbeast dan melihat visualisasi detail gambar berdampingan dengan grafik horizontal tingkat keyakinan (*confidence bar chart*).

In [ ]:
def test_random_thunderbeast_images(num_samples=3):
    """Mengambil sampel acak dari dataset Thunderbeast dan menampilkan visualisasi detail."""
    random_indices = random.sample(range(len(test_dataset)), num_samples)

    for idx in random_indices:
        item = test_dataset[idx]
        img = item["image"]
        true_lbl = item["label"]
        true_text = "AI-Generated" if true_lbl == 0 else "Real Image"

        # Preprocessing & Inference
        if img.mode == 'P' and 'transparency' in img.info:
            img = img.convert('RGBA')
        rgb_img = img.convert("RGB")
        tensor_img = eval_transforms(rgb_img).unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            outputs = model(tensor_img)
            probs = torch.softmax(outputs, dim=1)[0]
            conf, pred_idx = torch.max(probs, 0)

        p_idx = pred_idx.item()
        p_text = "AI-Generated" if p_idx == 0 else "Real Image"
        p_ai = probs[0].item() * 100.0
        p_real = probs[1].item() * 100.0
        c_pct = conf.item() * 100.0

        is_ok = (true_lbl == p_idx)
        status_s = "✅ BENAR" if is_ok else "❌ SALAH"
        color_v = "#059669" if is_ok else "#DC2626"

        print("=" * 65)
        print(f"🖼️  Sampel Thunderbeast  : Index #{idx} ({img.width}x{img.height} px)")
        print(f"🎯  Ground Truth         : {true_text}")
        print(f"🤖  Hasil Prediksi       : {p_text} [{status_s}]")
        print(f"📊  Keyakinan / Conf     : {c_pct:.2f}%")
        print(f"    ├─ Probabilitas AI   : {p_ai:.2f}%")
        print(f"    └─ Probabilitas Real : {p_real:.2f}%")
        print("=" * 65)

        fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={'width_ratios': [1.2, 1]})

        ax_img.imshow(img)
        ax_img.axis('off')
        ax_img.set_title(
            f"Index #{idx} | Hasil: {p_text} [{status_s}]
Ground Truth: {true_text} | Conf: {c_pct:.1f}%",
            color=color_v,
            fontsize=11.5,
            fontweight='bold',
            pad=8
        )

        classes = ['AI-Generated', 'Real Image']
        probabilities = [p_ai, p_real]
        bar_colors = ['#EF4444', '#10B981']

        bars = ax_bar.barh(classes, probabilities, color=bar_colors, height=0.5, edgecolor='#1E293B', linewidth=0.8)
        ax_bar.set_xlim(0, 100)
        ax_bar.set_xlabel('Probabilitas (%)', fontsize=10, fontweight='bold')
        ax_bar.set_title('Distribusi Keyakinan Model', fontsize=11, fontweight='bold', pad=8)
        ax_bar.grid(axis='x', linestyle='--', alpha=0.5)

        for bar in bars:
            w = bar.get_width()
            ax_bar.text(w + 2, bar.get_y() + bar.get_height() / 2, f"{w:.1f}%",
                        va='center', ha='left', fontsize=10, fontweight='bold', color='#1E293B')

        plt.tight_layout()
        plt.show()

# Jalankan pengujian 3 sampel acak
test_random_thunderbeast_images(num_samples=3)

---
## 💡 Panduan Penyesuaian Pengujian:

1. **Menguji Lebih Banyak Gambar**:
   Ubah variabel `NUM_TEST_SAMPLES` pada **Bagian 1**, misalnya:
   ```python
   NUM_TEST_SAMPLES = 5000  # atau len(test_dataset) untuk seluruh 15.271 gambar uji!
   ```
2. **Mengganti Model Checkpoint**:
   Anda dapat mengganti model aktif dengan memanggil:
   ```python
   model, active_model_name = load_model_checkpoint("EfficientNetV2-S")
   # Pilihan: 'ConvNeXt-Tiny', 'DINOv2-Small', 'EfficientNetV2-S', 'AIForensicNet'
   ```
3. **Menguji Indeks Tertentu**:
   Gunakan fungsi `test_random_thunderbeast_images(num_samples=5)` atau akses langsung `test_dataset[nomor_indeks]`.